# compare molskill sascore qed with lacan
Sample 10000 random molecules from ChEMBL36, assess them with QED,SA,Lacan and MolSkill

In [ ]:
import csv
from rdkit import Chem
smis = []
with open('data/chembl35_random_10K.csv', newline='') as csvfile:
    reader = csv.reader(csvfile, delimiter=',', quotechar='"')
    for row in reader:
        smis.append(row[0])
mols = [Chem.MolFromSmiles(smi) for smi in smis]

In [ ]:
from molskill.scorer import MolSkillScorer
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') #needed because molskill uses the old morgan api
scorer = MolSkillScorer()
molskill_scores = scorer.score(smis)

In [ ]:
import sys
import os
from rdkit.Chem import QED

sys.path.append(os.path.join(os.environ['CONDA_PREFIX'],'share','RDKit','Contrib'))

from SA_Score import sascorer
sa_scores = [sascorer.calculateScore(m) for m in mols]
qed_scores = [QED.qed(m) for m in mols]

In [ ]:
from lacan import lacan
p = lacan.load_profile("chembl_full")
lacan_scores = [lacan.score_mol(m,p)[0] for m in mols]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_style("white")
sns.set_context("paper", font_scale=1.2)

data = pd.DataFrame({
    "LACAN": lacan_scores,
    "SAScore": sa_scores,
    "QED": qed_scores,
    "MolSkill": molskill_scores
})

# --- Plotting functions ---

def lower_scatter_kde(x, y, **kwargs):
    ax = plt.gca()
    ax.scatter(x, y, alpha=0.08, s=4, color="steelblue", rasterized=True)
    sns.kdeplot(
        x=x, y=y, ax=ax,
        levels=5, thresh=0.1, bw_adjust=0.9, cut=0,
        color="navy", linewidths=0.8, fill=False,
    )
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() > 5:
        r, _ = spearmanr(x[mask], y[mask])


def upper_corr_tile(x, y, **kwargs):
    ax = plt.gca()
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() > 5:
        r, pval = spearmanr(x[mask], y[mask])
    else:
        r, pval = 0, 1
    cmap = plt.cm.RdBu_r
    color = cmap((r + 1) / 2)
    ax.set_facecolor(color)
    fontcolor = "white" if abs(r) > 0.4 else "black"
    ax.text(0.5, 0.5, f"ρ = {r:.2f}", transform=ax.transAxes,
            ha="center", va="center", fontsize=11, color=fontcolor,
            fontweight="bold")

def diag_hist(x, **kwargs):
    ax = plt.gca()
    sns.kdeplot(x, ax=ax, color="steelblue", fill=True, alpha=0.3,
                linewidth=1.2, bw_adjust=0.9, cut=0)

# --- Build PairGrid ---
g = sns.PairGrid(data, height=2.5, diag_sharey=False)

# Precompute consistent limits for each variable
lims = {}
for col in data.columns:
    v = data[col].values
    v = v[np.isfinite(v)]
    pad = 0.02 * (v.max() - v.min())
    lims[col] = (v.min() - pad, v.max() + pad)

# Unshare axes immediately after construction
for ax in g.axes.flatten():
    if ax is not None:
        ax.get_shared_x_axes().remove(ax)
        ax.get_shared_y_axes().remove(ax)

g.map_lower(lower_scatter_kde)
g.map_diag(diag_hist)
g.map_upper(upper_corr_tile)
for i, row_var in enumerate(data.columns):
    for j, col_var in enumerate(data.columns):
        ax = g.axes[i, j]
        if ax is None:
            continue

        # x corresponds to column variable
        ax.set_xlim(lims[col_var])

        # y corresponds to row variable (only for non-diagonal)
        if i != j:
            ax.set_ylim(lims[row_var])

# Clean up spines
for ax in g.axes.flatten():
    if ax is not None:
        ax.spines[["top", "right"]].set_visible(False)


g.figure.suptitle("Molecular Filter Correlations", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig("mol_filters.pdf", dpi=150, bbox_inches="tight")
plt.show()